# Money Connectome 01 - build the graph

**Milestone M0/M1.** Load the IBM synthetic AML transactions (HI-Small: ~5.08M
transactions, ~515k accounts, ~0.1% laundering), sanity-check them, and write the
aggregated edge list that notebooks 02 (metrics) and 03 (detection) read.

The package computes, this notebook orchestrates: all logic lives in
[`moneyconn`](https://github.com/gaaprojects/FlyAML) and is unit-tested on toy graphs
locally. Nothing here touches the dev laptop.

**Setup:** attach `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`,
CPU accelerator, internet **on** (needed for the `pip install` from GitHub).

**P0 acceptance criteria**
- [ ] loader reads ~5.08M rows
- [ ] laundering share ~0.1%
- [ ] peak RAM logged (budget: <= 20 GB)
- [ ] degree distributions plotted

## 1. Install the package

Pinned to a tag so a notebook run is reproducible. Bump the tag whenever the
package gains code this notebook needs.

In [ ]:
# Bump the tag whenever moneyconn gains code this notebook needs.
%pip install -q "git+https://github.com/gaaprojects/FlyAML.git@v0.1.0"
# While iterating on the package, install the branch instead of a tag:
# %pip install -q "git+https://github.com/gaaprojects/FlyAML.git@Develop"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import moneyconn
from moneyconn import load

log = load.configure_logging()
print("moneyconn", moneyconn.__version__, "| pandas", pd.__version__)

## 2. Find the attached dataset

Searched rather than hard-coded, so a re-attached or re-versioned dataset does not
break the run. The data is never copied or re-uploaded - only small derived
aggregates leave this notebook.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
VARIANT = "HI-Small"  # LI-Small is the P2 robustness check


def find_one(pattern: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(pattern))
    if not matches:
        raise FileNotFoundError(
            f"{pattern} not found under {INPUT_ROOT}. Attach the dataset "
            "'ealtman2019/ibm-transactions-for-anti-money-laundering-aml'."
        )
    return matches[0]


TRANS_PATH = find_one(f"{VARIANT}_Trans.csv")
PATTERNS_PATH = find_one(f"{VARIANT}_Patterns.txt")
print(TRANS_PATH, f"{TRANS_PATH.stat().st_size / 1024**3:.2f} GiB")
print(PATTERNS_PATH, f"{PATTERNS_PATH.stat().st_size / 1024**2:.1f} MiB")

## 3. Load the transactions

Compact dtypes (`category` / `float32` / `int8`) and account id = `f"{bank}_{account}"`.
Set `SMOKE_ROWS` to a few hundred thousand for a fast pass while editing; `None`
loads the whole file.

In [ ]:
SMOKE_ROWS = None  # e.g. 500_000 while iterating

with load.log_step(f"load {TRANS_PATH.name}"):
    tx = load.load_transactions(TRANS_PATH, nrows=SMOKE_ROWS)

tx.head()

In [ ]:
tx.dtypes.to_frame("dtype").assign(
    memory_mb=(tx.memory_usage(deep=True) / 1024**2).round(1)
)

## 4. Acceptance criteria

Expected on the full HI-Small file: ~5,078,345 transactions, ~515k accounts,
laundering share ~0.1%, ~10 simulated days.

In [ ]:
summary = load.summarize_transactions(tx)
for key, value in summary.items():
    print(f"{key:>24}: {value}")

In [ ]:
if SMOKE_ROWS is None:
    assert 5.0e6 < summary["transactions"] < 5.2e6, summary["transactions"]
    assert 4.0e5 < summary["accounts"] < 6.0e5, summary["accounts"]
    assert 5e-4 < summary["laundering_share"] < 5e-3, summary["laundering_share"]
    print("P0 acceptance criteria: OK")
else:
    print(f"smoke run over {SMOKE_ROWS:,} rows - assertions skipped")

## 5. Laundering patterns

Most laundering transactions carry no pattern label; only the rows in the patterns
file do. Per-pattern analysis (notebook 03) uses these rows only, while the overall
metrics use every `is_laundering` label.

In [ ]:
with load.log_step(f"parse {PATTERNS_PATH.name}"):
    patterns = load.parse_patterns(PATTERNS_PATH)

counts = load.pattern_counts(patterns)
print(f"{len(patterns):,} pattern transactions in {patterns['attempt_id'].nunique():,} attempts")
print(
    f"{len(patterns) / max(summary['laundering_transactions'], 1):.1%} "
    "of laundering transactions are pattern-assigned"
)
counts

## 6. Degree distributions

Transaction counts per account, not counterparty counts - the counterparty degrees
come from the aggregated graph in section 7. Log-log axes because a handful of
mega-hub accounts dominate; they are also what makes the nulls and motifs slow
later on (hence the hub-capped fallback in `moneyconn.nulls`).

In [ ]:
with load.log_step("per-account transaction counts"):
    activity = load.account_tx_counts(tx)

display(activity.head(10))
activity.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, column in zip(axes, ["sent", "received"]):
    freq = activity[column].value_counts().sort_index()
    freq = freq[freq.index > 0]
    ax.scatter(freq.index, freq.to_numpy(), s=8)
    ax.set(
        xscale="log",
        yscale="log",
        xlabel=f"transactions {column} per account",
        ylabel="accounts",
        title=f"{column.capitalize()} ({VARIANT})",
    )
    ax.grid(alpha=0.3, which="both")

fig.suptitle("Per-account transaction counts", y=1.02)
fig.tight_layout()

FIG_DIR = Path("/kaggle/working/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / "p0_transaction_counts.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Aggregate to a weighted edge list -- M1, not yet implemented

`moneyconn.graph` is still a stub. Once it lands (plan P1), this section collapses
repeated payments into one directed edge per account pair (`count`, `total_paid`,
`first_ts`, `last_ts`), writes the parquet checkpoint that notebooks 02 and 03 read,
and keeps the raw timestamped edge list for the time-respecting cycle search.

Acceptance criteria for M1: ~515k vertices and `edges["count"].sum() == len(tx)`.

In [ ]:
# Uncomment once graph.py is implemented (plan P1 / M1).
# from moneyconn import graph
#
# CHECKPOINT = Path("/kaggle/working/edges_hi_small.parquet")
#
# with load.log_step("aggregate edges"):
#     edges = graph.aggregate_edges(tx)
#
# assert edges["count"].sum() == len(tx), "edge counts must sum to the transaction count"
# graph.write_edges(edges, CHECKPOINT)
# print(f"{len(edges):,} edges -> {CHECKPOINT}")
#
# with load.log_step("build igraph"):
#     g = graph.build_graph(edges)
# print(g.summary())

## 8. Wrap up

Runtime and peak RAM for every step are in the log output above (goal G1: full
HI-Small pipeline <= 60 min, peak RAM <= 20 GB).

**Next:** save the aggregated edge-list parquet as a notebook output, then
`kaggle_02_metrics` reads that checkpoint and computes the connectome statistics
with degree-preserving nulls.

*Data: IBM synthetic AML transactions (Altman et al., 2023), used from the attached
Kaggle dataset and never re-uploaded. Synthetic data - no conclusions about real
people or institutions.*

In [ ]:
print("peak RAM this session:", f"{load.peak_ram_mb():,.0f} MiB")